In [0]:
hchb_ar_nonaggregated_history = dbutils.widgets.get("hchb_ar_nonaggregated_history")
mart_hchb_Ar = dbutils.widgets.get("mart_hchb_Ar")
hchb_ar_aggregate = dbutils.widgets.get("hchb_ar_aggregate")
hchb_ar_history = dbutils.widgets.get("hchb_ar_history")
mart_hchb_ar_aggregate = dbutils.widgets.get("mart_hchb_ar_aggregate")
fact_hchb_ar = dbutils.widgets.get("fact_hchb_ar")
fact_ar_staging = dbutils.widgets.get("fact_ar_staging")
mart_fact_ar = dbutils.widgets.get("mart_fact_ar")
date = dbutils.widgets.get("date")
officemapping = dbutils.widgets.get("officemapping")
client_episodes_all = dbutils.widgets.get("client_episodes_all")
client = dbutils.widgets.get("client")
payerdimension = dbutils.widgets.get("payerdimension")
cubeserviceofficetxnsourcesystem = dbutils.widgets.get("cubeserviceofficetxnsourcesystem")
office = dbutils.widgets.get("office")
cubeserviceofficetxnweekendingdate = dbutils.widgets.get("cubeserviceofficetxnweekendingdate")
alphacollector_claims = dbutils.widgets.get("alphacollector_claims")
atbfile = dbutils.widgets.get("atbfile")
tempdb_dbo_hchbar = dbutils.widgets.get("tempdb_dbo_hchbar")
staging_fact_hchb_bkp = dbutils.widgets.get("staging_fact_hchb_bkp")
ar_bkp=dbutils.widgets.get("ar_bkp")

In [0]:
# Truncate the temp table
spark.sql(f"TRUNCATE TABLE {fact_ar_staging}")
spark.sql(f"TRUNCATE TABLE {hchb_ar_aggregate}")

In [0]:
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW var_today AS
SELECT DATE_SUB(CAST(CURRENT_DATE() AS DATE), 8) AS today
""")

today_value = spark.sql("SELECT today FROM var_today").collect()[0]['today']
print(f"Reference date (today - 8): {today_value}")

# Step 2: Get current quarter number
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW var_CurrentQuarterNumber AS
SELECT dd.FiscalQuarterNbr AS CurrentQuarterNumber
FROM {date} dd
CROSS JOIN var_today t
WHERE dd.CalendarDate = t.today
LIMIT 1
""")

quarter_number = spark.sql("SELECT CurrentQuarterNumber FROM var_CurrentQuarterNumber").collect()[0]['CurrentQuarterNumber']
print(f"Current quarter number: {quarter_number}")

# Step 3: Get end of current quarter (max week ending date for the quarter)
end_of_quarter_result = spark.sql(f"""
SELECT MAX(CAST(dd.WeekEndingDate AS TIMESTAMP)) AS EndOfCurrentQuarter
FROM {date} dd
CROSS JOIN var_CurrentQuarterNumber cqn
WHERE dd.FiscalQuarterNbr = cqn.CurrentQuarterNumber
""")

end_of_current_quarter = end_of_quarter_result.collect()[0]['EndOfCurrentQuarter']
print(f"End of current quarter: {end_of_current_quarter}")

print("✓ Quarter end calculation completed")

In [0]:

spark.sql(f"""
INSERT INTO {hchb_ar_nonaggregated_history} (
    reporting_week_ending_date,
    net_earned_ar,
    gross_ar,
    days_0_90,
    days_91_180,
    days_181_270,
    days_271_360,
    days_361_plus,
    episode_in_progress,
    cash,
    unearned_rev,
    adjustments,
    earned_rev,
    revenue,
    reporting_branchcode,
    bill_date,
    payor_name,
    psid,
    payor_type,
    paid,
    client_last_name,
    end_date,
    key_date,
    pps

)
SELECT
    reporting_week_ending_date,
    net_earned_ar,
    gross_ar,
    days_0_90,
    days_91_180,
    days_181_270,
    days_271_360,
    days_361_plus,
    episode_in_progress,
    cash,
    unearned_rev,
    adjustments,
    earned_rev,
    revenue,
    reporting_branchcode,
    bill_date,
    payor_name,
    psid,
    payor_type,
    paid,
    client_last_name,
    end_date,
    key_date,
    pps
FROM {tempdb_dbo_hchbar}
""")

# Verify insert
count = spark.sql(f"SELECT COUNT(*) as count FROM {hchb_ar_nonaggregated_history}").collect()[0]['count']
print(f"✓ Inserted into hchb_ar_nonaggregated_history: {count} total records")

In [0]:
# ========================================
# INSERT INTO MART HCHB AR
# ========================================

print("Inserting data into mart HCHB AR...")

spark.sql(f"""
INSERT INTO {mart_hchb_Ar} (
    reporting_week_ending_date,
    net_earned_ar,
    gross_ar,
    days_0_90,
    days_91_180,
    days_181_270,
    days_271_360,
    days_361_plus,
    episode_in_progress,
    cash,
    unearned_rev,
    adjustments,
    earned_rev,
    revenue,
    reporting_branchcode,
    bill_date,
    payor_name,
    psid,
    payor_type,
    paid,
    client_last_name,
    end_date,
    key_date,
    pps
)
SELECT
    reporting_week_ending_date,
    net_earned_ar,
    gross_ar,
    days_0_90,
    days_91_180,
    days_181_270,
    days_271_360,
    days_361_plus,
    episode_in_progress,
    cash,
    unearned_rev,
    adjustments,
    earned_rev,
    revenue,
    reporting_branchcode,
    bill_date,
    payor_name,
    psid,
    payor_type,
    paid,
    client_last_name,
    end_date,
    key_date,
    pps
FROM {tempdb_dbo_hchbar}
""")

# Verify insert
count = spark.sql(f"SELECT COUNT(*) as count FROM {mart_hchb_Ar}").collect()[0]['count']
print(f"✓ Inserted into mart_hchb_Ar: {count} total records")

In [0]:

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW ar_dataset AS
SELECT
    a.reporting_week_ending_date,
    a.source_system,
    a.pps,
    a.key_date,
    a.end_date,
    a.client_last_name,
    a.paid,
    a.payor_type,
    a.psid,
    a.payor_name,
    a.bill_date,
    a.reporting_branchcode,
    a.revenue,
    a.earned_rev,
    a.adjustments,
    a.unearned_rev,
    a.cash,
    a.episode_in_progress,
    a.collector_name,
    a.ar_0_90_days,
    a.ar_91_180_days,
    a.ar_181_270_days,
    a.ar_271_plus_days,
    CASE 
        WHEN A.reporting_branchcode  RLIKE '[A-Z]'
            THEN OM.TargetOfficeNumber
        ELSE CAST(A.reporting_branchcode AS INT)
    END AS TargetOfficeNumber,
    CONCAT(
        COALESCE(CAST(A.paid AS STRING), ''),
        ' - ',
        COALESCE(CAST(A.paid AS STRING), '')
    ) AS invoice_number
FROM (
    SELECT 
         reporting_week_ending_date,
        CAST('HCHB' AS STRING) AS source_system,
        pps,
        key_date,
        end_date,
        client_last_name,
        paid,
        payor_type,
        psid,
        payor_name,
        bill_date,
        reporting_branchcode,
        revenue,
        earned_rev,
        adjustments,
        unearned_rev,
        cash,
        episode_in_progress,
        CAST('' AS STRING) AS collector_name,
        SUM(COALESCE(days_0_90, 0))        AS ar_0_90_days,
        SUM(COALESCE(days_91_180, 0))      AS ar_91_180_days,
        SUM(COALESCE(days_181_270, 0))     AS ar_181_270_days,
        SUM(COALESCE(days_271_360, 0))
          + SUM(COALESCE(days_361_plus, 0)) AS ar_271_plus_days
    FROM {tempdb_dbo_hchbar}
    GROUP BY 
        reporting_week_ending_date,
        pps,
        key_date,
        end_date,
        client_last_name,
        paid,
        payor_type,
        psid,
        payor_name,
        bill_date,
        reporting_branchcode,
        revenue,
        earned_rev,
        adjustments,
        unearned_rev,
        cash,
        episode_in_progress
) A
LEFT JOIN {officemapping} OM
    ON OM.SourceOfficeCode = A.Reporting_Branchcode
""")

ar_count = spark.sql("SELECT COUNT(*) as count FROM ar_dataset").collect()[0]['count']

In [0]:
spark.sql(f"""
INSERT INTO {hchb_ar_aggregate}
SELECT 
    reporting_week_ending_date        AS reporting_week_ending_date,
    CAST('HCHB' AS STRING)            AS source_system,
    client_last_name                  AS client_last_name,
    paid                              AS client_number,
    payor_type                        AS payor_type,
    psid                              AS payor_number,
    payor_name                        AS payor_name,
    collector_name                    AS collector_name,
    ar_0_90_days                      AS ar_0_90_days,
    ar_91_180_days                    AS ar_91_180_days,
    ar_181_270_days                   AS ar_181_270_days,
    ar_271_plus_days                  AS ar_271_plus_days,
    TargetOfficeNumber                AS service_office_number,
    invoice_number                    AS invoice_number
FROM ar_dataset
""");

In [0]:
spark.sql(f"""
INSERT INTO {hchb_ar_history}
SELECT 
    reporting_week_ending_date        AS reporting_week_ending_date,
    CAST('HCHB' AS STRING)            AS source_system,
    client_last_name                  AS client_last_name,
    paid                              AS client_number,
    payor_type                        AS payor_type,
    psid                              AS payor_number,
    payor_name                        AS payor_name,
    collector_name                    AS collector_name,
    ar_0_90_days                      AS ar_0_90_days,
    ar_91_180_days                    AS ar_91_180_days,
    ar_181_270_days                   AS ar_181_270_days,
    ar_271_plus_days                  AS ar_271_plus_days,
    TargetOfficeNumber                AS service_office_number,
    invoice_number                    AS invoice_number
FROM ar_dataset
""");

In [0]:
spark.sql(f"""
INSERT INTO {mart_hchb_ar_aggregate}
SELECT 
    reporting_week_ending_date        AS reporting_week_ending_date,
    CAST('HCHB' AS STRING)            AS source_system,
    client_last_name                  AS client_last_name,
    paid                              AS client_number,
    payor_type                        AS payor_type,
    psid                              AS payor_number,
    payor_name                        AS payor_name,
    collector_name                    AS collector_name,
    ar_0_90_days                      AS ar_0_90_days,
    ar_91_180_days                    AS ar_91_180_days,
    ar_181_270_days                   AS ar_181_270_days,
    ar_271_plus_days                  AS ar_271_plus_days,
    TargetOfficeNumber                AS service_office_number,
    invoice_number                    AS invoice_number
FROM ar_dataset
""");

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW ar_fact_dataset AS
SELECT
    ar.*,
    csi.SourceSystemId,
    csi.ClientKey,
    o.OfficeKey,
    W.WeekEndingDateKey AS reporting_weekending_date_key,
    p.PayerKey,
    S.SourceSystemKey
FROM (
    SELECT
        A.*,
        hcea.epi_id
    FROM {hchb_ar_aggregate} A

    LEFT JOIN (
        SELECT epi_id, epi_paid
        FROM (
            SELECT
                epi_id,
                epi_paid,
                ROW_NUMBER() OVER (
                    PARTITION BY epi_paid
                    ORDER BY epi_paid
                ) AS rnk
            FROM {client_episodes_all}
        ) hca1
        WHERE rnk = 1
    ) hcea
        ON A.client_number = CAST(hcea.epi_paid AS STRING)
) ar
LEFT JOIN (
    SELECT
        SourceSystemId,
        ClientKey,
        SourceSystem
    FROM (
        SELECT
            rc.SourceSystemId,
            rc.ClientKey,
            rc.SourceSystem,
            ROW_NUMBER() OVER (
                PARTITION BY rc.SourceSystemId, rc.SourceSystem
                ORDER BY rc.SourceSystemId
            ) AS rnk1
        FROM {client} rc
    ) rc1
    WHERE rnk1 = 1
) csi
    ON CAST(ar.epi_id AS STRING) = TRIM(CAST(csi.SourceSystemId AS STRING))
   AND csi.SourceSystem = 'HCHB'

LEFT JOIN {payerdimension} p
    ON CAST(ar.Payor_number AS STRING) = CAST(p.PayerID AS STRING)

LEFT JOIN {office} o
    ON o.OfficeNumber = ar.service_office_number

LEFT JOIN {cubeserviceofficetxnsourcesystem} S
    ON S.SourceSystemName = ar.source_system

LEFT JOIN {cubeserviceofficetxnweekendingdate} W
    ON W.WeekEndingDate = CAST(ar.reporting_week_ending_date AS DATE)
""")

fact_count = spark.sql("SELECT COUNT(*) as count FROM ar_fact_dataset").collect()[0]['count']

In [0]:
spark.sql(f"""
INSERT INTO {fact_hchb_ar} (
    reporting_weekending_date_key,
    source_system_key,
    collector_name,
    ar_0_90_days,
    ar_91_180_days,
    ar_181_270_days,
    ar_271_plus_days,
    invoice_number,
    client_key,
    office_key,
    payor_key,
    payor_type
)
SELECT  
    reporting_weekending_date_key AS reporting_weekending_date_key,
    SourceSystemKey                    AS source_system_key,
    collector_name                        AS collector_name,
    ar_0_90_days,
    ar_91_180_days,
    ar_181_270_days,
    ar_271_plus_days,
    invoice_number,
    ClientKey                          AS client_key,
    OfficeKey                          AS office_key,
    PayerKey                           AS payor_key,
    payor_type                          AS payor_type
FROM ar_fact_dataset
""")
count = spark.sql(f"SELECT COUNT(*) as count FROM {fact_hchb_ar}").collect()[0]['count']


In [0]:
spark.sql(f"""
INSERT INTO {fact_ar_staging} (
    reporting_weekending_date_key,
    source_system_key,
    collector_name,
    ar_0_90_days,
    ar_91_180_days,
    ar_181_270_days,
    ar_271_plus_days,
    invoice_number,
    client_key,
    office_key,
    payor_key,
    payor_type
)
SELECT  
    reporting_weekending_date_key AS reporting_weekending_date_key,
    SourceSystemKey                    AS source_system_key,
    collector_name                        AS collector_name,
    ar_0_90_days,
    ar_91_180_days,
    ar_181_270_days,
    ar_271_plus_days,
    invoice_number,
    ClientKey                          AS client_key,
    OfficeKey                          AS office_key,
    PayerKey                           AS payor_key,
    payor_type                          AS payor_type
FROM ar_fact_dataset
""")

# Verify insert
count = spark.sql(f"SELECT COUNT(*) as count FROM {fact_ar_staging}").collect()[0]['count']
print(f"✓ Inserted into fact_ar_staging: {count} total records")

# Show AR distribution across aging buckets
print("\nAR Distribution Across Aging Buckets:")
spark.sql(f"""
SELECT 
    SUM(ar_0_90_days) as total_0_90,
    SUM(ar_91_180_days) as total_91_180,
    SUM(ar_181_270_days) as total_181_270,
    SUM(ar_271_plus_days) as total_271_plus,
    SUM(ar_0_90_days + ar_91_180_days + ar_181_270_days + ar_271_plus_days) as grand_total
FROM {fact_ar_staging}
""").show(truncate=False)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW ar_fact AS
SELECT
    CASE
        WHEN (
            COALESCE(ar_0_90_days, 0)
          + COALESCE(ar_91_180_days, 0)
          + COALESCE(ar_181_270_days, 0)
          + COALESCE(ar_271_plus_days, 0)
        ) < 0
        THEN 'Credit'
        ELSE 'Debit'
    END AS invoice_balance_type,

    reporting_weekending_date_key,
    payor_key,
    source_system_key,
    client_key,
    office_key,
    payor_type            AS payor_type_code,
    collector_name,
    ar_0_90_days,
    ar_91_180_days,
    ar_181_270_days,
    ar_271_plus_days,
    invoice_number
FROM {fact_ar_staging} A
""")

fact_count = spark.sql("SELECT COUNT(*) as count FROM ar_fact").collect()[0]['count']

In [0]:
count = spark.sql(f"SELECT COUNT(*) as cnt FROM {mart_fact_ar}").collect()[0]['cnt']
if count == 0:
    print(f"Full load for {mart_fact_ar} executed")
    spark.sql(f"""
    INSERT INTO {mart_fact_ar} (
        invoice_balance_type,
        reporting_weekending_date_key,
        source_system_key,
        client_key,
        office_key,
        payor_key,
        payor_type,
        collector_name,
        ar_0_90_days,
        ar_91_180_days,
        ar_181_270_days,
        ar_271_plus_days,
        invoice_number
    )

    SELECT
        CAST(Invoice_Balance_Type AS STRING) as invoice_balance_type,
        CAST(Reporting_Weekending_Date_Key AS int) as reporting_weekending_date_key,
        CAST(Source_System_Key AS int) as source_system_key,
        CAST(Client_Key AS int) as client_key,
        CAST(Office_Key AS int) as office_key,
        CAST(Payor_Key AS int) as payor_key,
        CAST(Payor_Type_Code AS string) as payor_type,
        CAST(Collector_Name AS STRING) as collector_name,
        CAST(AR_0_90_Days AS double) as ar_0_90_days,
        CAST(AR_91_180_Days AS double) as ar_91_180_days,
        CAST(AR_181_270_Days AS double) as ar_181_270_days,
        CAST(AR_271___Days AS double) as ar_271_plus_days,
        CAST(Invoice_Number AS STRING) as invoice_number
    FROM {ar_bkp}
    WHERE source_system_key = '0' OR source_system_key = '19'

    UNION ALL

    SELECT
        CASE
            WHEN (
                COALESCE(AR_0_90_Days, 0)
              + COALESCE(AR_91_180_Days, 0)
              + COALESCE(AR_181_270_Days, 0)
              + COALESCE(AR_271___Days, 0)
            ) < 0
            THEN 'Credit'
            ELSE 'Debit'
        END AS invoice_balance_type,
        CAST(Reporting_Weekending_Date_Key AS int) as reporting_weekending_date_key,
        CAST(Source_System_Key AS int) as source_system_key,
        CAST(Client_Key AS int) as client_key,
        CAST(Office_Key AS int) as office_key,
        CAST(Payor_Key AS int) as payor_key,
        CAST(Payor_Type AS string) as payor_type,
        CAST(Collector_Name AS STRING) as collector_name,
        CAST(AR_0_90_Days AS double) as ar_0_90_days,
        CAST(AR_91_180_Days AS double) as ar_91_180_days,
        CAST(AR_181_270_Days AS double) as ar_181_270_days,
        CAST(AR_271___Days AS double) as ar_271_plus_days,
        CAST(Invoice_Number AS STRING) as invoice_number
    FROM {staging_fact_hchb_bkp}
    WHERE source_system_key = '6'
    """)
else:
    print("Skipping the full load")

In [0]:
spark.sql(f"""
INSERT INTO {mart_fact_ar} (
    invoice_balance_type,
    reporting_weekending_date_key,
    source_system_key,
    client_key,
    office_key,
    payor_key,
    payor_type,
    collector_name,
    ar_0_90_days,
    ar_91_180_days,
    ar_181_270_days,
    ar_271_plus_days,
    invoice_number
)
SELECT
    invoice_balance_type,
    reporting_weekending_date_key,
    source_system_key,
    client_key,
    office_key,
    payor_key,
    payor_type_code       AS payor_type,
    collector_name,
    ar_0_90_days,
    ar_91_180_days,
    ar_181_270_days,
    ar_271_plus_days,
    invoice_number
FROM ar_fact
""")

# Verify insert
count = spark.sql(f"SELECT COUNT(*) as count FROM {mart_fact_ar}").collect()[0]['count']
print(f"✓ Inserted into mart_fact_ar: {count} total records")

In [0]:
# ========================================
# CREATE AR FACT CUBHUB TEMP VIEW
# ========================================

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW ar_fact_cubhub AS
SELECT
    CASE
        WHEN (
            COALESCE(ar_0_90_days, 0)
          + COALESCE(ar_91_180_days, 0)
          + COALESCE(ar_181_270_days, 0)
          + COALESCE(ar_271_plus_days, 0)
        ) < 0
        THEN 'Credit'
        ELSE 'Debit'
    END AS invoice_balance_type,
    *
FROM (
    SELECT
        CAST(date_format(date_add(current_date(), -4), 'yyyyMMdd') AS INT)
            AS reporting_week_ending_date_key,

        pd.PayerKey        AS payor_key,
        19                 AS source_system_key,
        clt.ClientKey      AS client_key,
        ofc.OfficeKey      AS office_key,
        atb.AssignedTo     AS collector_name,
        clm.ClaimNumber    AS invoice_number,
        pd.TypeID          AS payor_type,

        /* AR buckets */
        CASE
            WHEN to_date(DateBilled, 'MM/dd/yyyy') <= current_date()
             AND to_date(DateBilled, 'MM/dd/yyyy') >= date_add(current_date(), -90)
            THEN COALESCE(Balance, 0)
        END AS ar_0_90_days,

        CASE
            WHEN to_date(DateBilled, 'MM/dd/yyyy') <= date_add(current_date(), -91)
             AND to_date(DateBilled, 'MM/dd/yyyy') >= date_add(current_date(), -180)
            THEN COALESCE(Balance, 0)
        END AS ar_91_180_days,

        CASE
            WHEN to_date(DateBilled, 'MM/dd/yyyy') <= date_add(current_date(), -181)
             AND to_date(DateBilled, 'MM/dd/yyyy') >= date_add(current_date(), -270)
            THEN COALESCE(Balance, 0)
        END AS ar_181_270_days,

        CASE
            WHEN to_date(DateBilled, 'MM/dd/yyyy') <= date_add(current_date(), -271)
            THEN COALESCE(Balance, 0)
        END AS ar_271_plus_days

    FROM {alphacollector_claims} clm

    /* Latest ATB record */
    LEFT JOIN (
        SELECT
            AccountNumber,
            Patient,
            FacilityCode,
            ActiveInsName,
            AssignedTo
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY AccountNumber, Facility
                    ORDER BY ReportingDate DESC
                ) AS rnb
            FROM {atbfile}
            WHERE (
                    AccountNumber RLIKE '[a-z][a-z]'
                 OR AccountNumber RLIKE '[a-z][a-z]s'
                  )
              AND AccountNumber NOT LIKE 'ADV%%'
              AND ActiveInsName <> ''
              AND ReportingDate >= DATE '2024-08-22'
        ) a
        WHERE rnb = 1
    ) atb
        ON atb.AccountNumber = clm.ClaimNumber
       AND atb.FacilityCode = clm.OfficeExternalId
       AND clm.PayerName = atb.ActiveInsName

    /* Latest Payer */
    LEFT JOIN (
        SELECT
            PayerKey,
            Name,
            TypeID
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY Name
                    ORDER BY PayerKey DESC
                ) AS rnb
            FROM {payerdimension}
            WHERE SourceSystemKey = 19
        ) p
        WHERE rnb = 1
    ) pd
        ON pd.Name = atb.ActiveInsName

    /* Latest Client */
    LEFT JOIN (
        SELECT
            ClientKey,
            MedicalRecordNumber
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY MedicalRecordNumber
                    ORDER BY ClientKey DESC
                ) AS rnb
            FROM {client}
            WHERE SourceSystem = 'CubHub'
        ) c
        WHERE rnb = 1
    ) clt
        ON clm.MedicalRecordNumber = clt.MedicalRecordNumber

    LEFT JOIN {office} ofc
        ON ofc.OfficeNumber = clm.OfficeExternalId
) a
""")

cubhub_count = spark.sql("SELECT COUNT(*) as count FROM ar_fact_cubhub").collect()[0]['count']
print(f"✓ AR fact CubHub temp view created: {cubhub_count} records")


In [0]:
spark.sql(f"""
INSERT INTO {mart_fact_ar} (
    invoice_balance_type,
    reporting_weekending_date_key,
    source_system_key,
    client_key,
    office_key,
    payor_key,
    payor_type,
    collector_name,
    ar_0_90_days,
    ar_91_180_days,
    ar_181_270_days,
    ar_271_plus_days,
    invoice_number
)
SELECT
    invoice_balance_type,
    reporting_week_ending_date_key,
    source_system_key,
    client_key,
    office_key,
    payor_key,
    payor_type  AS payor_type,
    collector_name,
    ar_0_90_days,
    ar_91_180_days,
    ar_181_270_days,
    ar_271_plus_days,
    invoice_number
FROM ar_fact_cubhub
""");
